# Quick Check: v3_2022_5m Run Results

**Date**: January 7, 2026  
**Run**: v3_2022_5m (2022+ data, 5m bars)

This notebook provides a quick sanity check of the v3_2022_5m pipeline results.

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

%matplotlib inline

RUN_DIR = Path("../runs/v3_2022_5m")
BAR_SIZE = "5m"

print(f"Checking run: {RUN_DIR}")
print(f"Bar size: {BAR_SIZE}")

Checking run: ../runs/v3_2022_5m
Bar size: 5m


## 1. K-Fold Backtest Results

In [2]:
# Load K-fold summary
kfold_path = RUN_DIR / f"bar_size={BAR_SIZE}" / "backtests" / "purged_kfold" / "summary.json"
kfold = json.load(open(kfold_path))

print("K-FOLD BACKTEST RESULTS")
print("=" * 60)

for fold in kfold['metrics_by_split']:
    print(f"Fold {fold['split_id']}: ${fold['total_pnl_usd']:>8,.0f}  WR: {fold['win_rate']:.1%}  PF: {fold['profit_factor']:.2f}  Trades: {fold['trades_count']}")

total_pnl = sum(f['total_pnl_usd'] for f in kfold['metrics_by_split'])
avg_wr = np.mean([f['win_rate'] for f in kfold['metrics_by_split']])

print("-" * 60)
print(f"TOTAL: ${total_pnl:>8,.0f}  Avg WR: {avg_wr:.1%}")
print("=" * 60)

K-FOLD BACKTEST RESULTS
Fold 0: $   2,769  WR: 53.5%  PF: 1.25  Trades: 260
Fold 1: $   4,383  WR: 59.9%  PF: 1.72  Trades: 152
Fold 2: $   3,650  WR: 52.4%  PF: 1.62  Trades: 103
Fold 3: $   2,660  WR: 58.5%  PF: 1.84  Trades: 53
Fold 4: $  15,390  WR: 72.8%  PF: 6.37  Trades: 81
Fold 5: $   8,606  WR: 61.0%  PF: 1.87  Trades: 154
------------------------------------------------------------
TOTAL: $  37,459  Avg WR: 59.7%


## 2. Walk-Forward Results

In [3]:
# Load walk-forward summary
wf_path = RUN_DIR / "walkforward" / f"bar_size={BAR_SIZE}" / "summary.json"
wf = json.load(open(wf_path))

print("WALK-FORWARD VALIDATION RESULTS")
print("=" * 60)

for window in wf['metrics_by_window']:
    status = "✓" if window['total_pnl_usd'] > 0 else "✗"
    print(f"Window {window['window_id']:2d}: ${window['total_pnl_usd']:>8,.0f}  Trades: {window['trades_count']:3d}  {status}")

wf_total = sum(w['total_pnl_usd'] for w in wf['metrics_by_window'])
profitable = sum(1 for w in wf['metrics_by_window'] if w['total_pnl_usd'] > 0)

print("-" * 60)
print(f"TOTAL: ${wf_total:>8,.0f}  Profitable: {profitable}/{len(wf['metrics_by_window'])}")
print("=" * 60)

WALK-FORWARD VALIDATION RESULTS
Window  0: $     467  Trades:  33  ✓
Window  1: $   2,077  Trades:  46  ✓
Window  2: $     706  Trades:  33  ✓
Window  3: $   2,446  Trades:  46  ✓
Window  4: $  -2,461  Trades:   6  ✗
Window  5: $   2,801  Trades:  35  ✓
Window  6: $   1,449  Trades:  42  ✓
Window  7: $     205  Trades:  45  ✓
Window  8: $   1,876  Trades:  32  ✓
Window  9: $   2,969  Trades:  39  ✓
Window 10: $   2,028  Trades:  40  ✓
Window 11: $    -117  Trades:  39  ✗
Window 12: $   1,467  Trades:  38  ✓
Window 13: $   3,838  Trades:  47  ✓
Window 14: $   2,498  Trades:  42  ✓
------------------------------------------------------------
TOTAL: $  22,249  Profitable: 13/15


## 3. Feature Completeness

In [4]:
# Check feature completeness
features = pd.read_parquet(RUN_DIR / f"bar_size={BAR_SIZE}" / "features.parquet")

print("FEATURE COMPLETENESS")
print("=" * 60)
print(f"Total rows: {len(features):,}")

if 'usable_for_training' in features.columns:
    usable_pct = features['usable_for_training'].mean()
    print(f"Usable for training: {usable_pct:.1%}")
else:
    print("'usable_for_training' column not found")

# Check for NaNs
nan_counts = features.isna().sum()
cols_with_nans = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(f"\nColumns with NaNs: {len(cols_with_nans)}")
if len(cols_with_nans) > 0:
    print("\nTop 10 columns with most NaNs:")
    for col, count in cols_with_nans.head(10).items():
        pct = 100 * count / len(features)
        print(f"  {col}: {count:,} ({pct:.1f}%)")

FEATURE COMPLETENESS
Total rows: 36,201
Usable for training: 99.8%

Columns with NaNs: 17

Top 10 columns with most NaNs:
  vol_regime: 69 (0.2%)
  price_vs_vwap: 49 (0.1%)
  trend_strength: 29 (0.1%)
  sma_30: 29 (0.1%)
  log_return_24: 24 (0.1%)
  vol_20: 20 (0.1%)
  parkinson_vol: 19 (0.1%)
  sma_20: 19 (0.1%)
  bb_position: 19 (0.1%)
  autocorr_5: 19 (0.1%)


## 4. Summary Comparison

In [5]:
print("SUMMARY COMPARISON")
print("=" * 60)
print(f"K-Fold Total PnL:       ${total_pnl:>10,.0f}")
print(f"Walk-Forward Total PnL: ${wf_total:>10,.0f}")
print(f"Ratio (WF/KF):          {wf_total/total_pnl:>10.1%}")
print()
print("Walk-forward retaining >50% of K-fold PnL is a good sign of generalization.")
print("=" * 60)

SUMMARY COMPARISON
K-Fold Total PnL:       $    37,459
Walk-Forward Total PnL: $    22,249
Ratio (WF/KF):               59.4%

Walk-forward retaining >50% of K-fold PnL is a good sign of generalization.


## 5. Training Metrics

In [6]:
# Check training metrics
training_dir = RUN_DIR / f"bar_size={BAR_SIZE}" / "training" / "purged_kfold"

print("TRAINING METRICS (ROC-AUC)")
print("=" * 60)

aucs = []
for fold_dir in sorted(training_dir.glob("fold_*")):
    metrics_path = fold_dir / "metrics.json"
    if metrics_path.exists():
        metrics = json.load(open(metrics_path))
        auc = metrics['metrics']['roc_auc_target_vs_rest']
        aucs.append(auc)
        print(f"{fold_dir.name}: {auc:.3f}")

if aucs:
    print("-" * 60)
    print(f"Mean: {np.mean(aucs):.3f}  Std: {np.std(aucs):.3f}")
    print(f"CV Stability (CoV): {np.std(aucs)/np.mean(aucs):.3f}")
    print("=" * 60)

TRAINING METRICS (ROC-AUC)
fold_0: 0.558
fold_1: 0.694
fold_2: 0.642
fold_3: 0.701
fold_4: 0.790
fold_5: 0.845
------------------------------------------------------------
Mean: 0.705  Std: 0.094
CV Stability (CoV): 0.133
